# Exercise 3: the prototype against the survey simulation 

*Covers notebook 3, exercise 2 and the ArgusSim survey tutorial
(`argus_survey_model/notebooks/tutorial_survey_data.ipynb`).*

ArgusSim predicts every exposure's noise for the Array as designed. The
prototype is real hardware on the real sky. Comparing the two is a sanity
check, not a validation: the prototype's cameras, site conditions and
reduction differ from the simulation's assumptions, and the simulation's
errors assume PSF-fitting photometry with a 5 mmag floor, where we used
apertures. The v5.4 run also predates ArgusSim's current filter curves,
which make a ρ exposure about 0.05 mag shallower. Differences are questions
to ask.

**Goal:** overlay the simulation's predicted ρ scatter for 60 s exposures
at segment 391749 on the prototype's noise curve from exercise 2.

This needs `argus_sim` (`pip install ./argus/argus_survey_model`) and
`noise_rho.ecsv` from exercise 2.

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.visualization import AsinhStretch, ImageNormalize, ZScaleInterval

import argex
from argex import sky, times

warnings.filterwarnings("ignore", category=RuntimeWarning)  # all-NaN slices
GREEK = {"beta": "β", "rho": "ρ"}  # band names in plot labels
plt.rcParams.update({"figure.dpi": 110, "image.origin": "lower"})

# The staged sample. Set ARGUS_SAMPLE to read another copy, such as a download.
SAMPLE = Path(os.environ.get("ARGUS_SAMPLE", "~/s3/schmidt-observatory-system/argus/prototype_v1")).expanduser()
assert (SAMPLE / "manifest.json").is_file(), f"no sample at {SAMPLE}: set ARGUS_SAMPLE"

from astropy.table import Table

In [ ]:
# The simulated 5-year survey (argus_survey_model/). Set ARGUS_SIM_STORE to read another copy.
import logging

from argus_sim.lightcurve import SurveyRun, bin_lightcurve

logging.getLogger("argus_sim").setLevel(logging.WARNING)
SIM_STORE = Path(os.environ.get(
    "ARGUS_SIM_STORE", "~/s3/schmidt-observatory-system/argus/sim/argsim_5yr_beta_rho_uniform_v5.4/store")).expanduser()
run = SurveyRun(str(SIM_STORE))

## 1. The simulation's exposures here

Get the simulated ρ visits at the segment's center (`sky.seg_center()`) and
keep the 60 s exposures (`exptime_s == 60`). How many are there in five
years, and what are their median airmass and seeing?

In [ ]:
ra_c, dec_c = sky.seg_center(391749)
vis = ...

## 2. The predicted scatter

`run.sigma_mag(vis, mag)` gives each exposure's predicted magnitude error
for a source of magnitude `mag`. Take its median over the exposures for a
range of magnitudes, and overlay it on the prototype's noise curve.

In [ ]:
proto = Table.read("noise_rho.ecsv")
mags = np.arange(9, 17.5, 0.25)
predicted = ...

## 3. Questions

* Where do the two agree, and by how much does the prototype exceed the
  prediction at the faint end?
* List what could explain the difference: aperture against PSF photometry,
  the sky on these particular nights (look at notebook 2's zeropoints),
  the camera, the interpolation in the reprojection.
* What would you measure next to tell those apart?